# Evaluating Retrieval Quality

**Purpose:** Think like researchers/engineers: define what “good” means and measure it.

In this notebook we will:
1. Define success criteria (relevance, speed, coverage).
2. Create a small labeled evaluation set (20 queries × top-10 results).
3. Compute common retrieval metrics:
   - **Precision@k**
   - **Recall@k** (when you have a ground-truth relevant set)
   - **MRR** (Mean Reciprocal Rank) — optional but simple and useful
4. Compare choices:
   - encoder A vs encoder B **or**
   - metric/index choice change
5. Interpret results: what improved, what got worse?
6. Produce:
   - a small metrics table
   - a before vs after comparison

---

> **Reminder:** Similarity is a *signal*. Evaluation tells you whether the signal is useful for your task.


## 0) Setup

We’ll use:
- sentence-transformers for embeddings
- FAISS for retrieval
- pandas for tables

We’ll compare two encoders:
- `all-MiniLM-L6-v2` (fast, common)
- `all-mpnet-base-v2` (often stronger but slower)

If you want to compare *metrics* instead, you can adapt the “system” definition later.


In [1]:
# Install dependencies if needed (safe to re-run)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    !uv add sentence-transformers

try:
    import faiss  # noqa: F401
except ImportError:
    !uv add faiss-cpu

In [2]:
import time
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 140)
np.random.seed(42)

## 1) Define success

Retrieval quality is not one thing.

Common dimensions:
- **Semantic relevance**: Are retrieved items truly relevant?
- **Speed/latency**: Is retrieval fast enough for the product?
- **Coverage/recall**: Does the system find the relevant stuff at all?
- **Stability**: Does it behave consistently across query styles?
- **Constraints**: Are filters (date, permissions, domain) respected?

In this notebook we focus on **relevance-oriented metrics**:
- Precision@k
- Recall@k
- MRR

We will also record approximate retrieval time for each system as an extra signal.


## 2) Build a dataset + queries

We’ll use a medium-small dataset of short snippets.
To keep this notebook self-contained, we generate a dataset across topics.

Then we define **20 evaluation queries** spanning those topics.

Finally, we create a *labeling workflow*:
- For each query, retrieve top-10 results
- Mark each as relevant (1) or not (0)
- Optionally mark “highly relevant” (2) if you want graded labels later


In [3]:
# A compact, diverse snippet dataset (~300)
topics = {
    "finance": [
        "Reduce spending by reviewing subscriptions and recurring bills.",
        "Build an emergency fund by setting aside a small amount weekly.",
        "Pay down high-interest debt first to save on interest payments.",
        "Banks assess credit risk before approving loans.",
        "Diversify investments to manage portfolio risk.",
        "Track your cash flow to understand where your money goes.",
    ],
    "software": [
        "Refactor code to reduce technical debt and improve maintainability.",
        "Profile your program to find performance bottlenecks.",
        "Add caching to avoid recomputing expensive results.",
        "Use pagination when retrieving large datasets from APIs.",
        "Improve reliability by adding retries and timeouts.",
        "Vectorize operations in NumPy for faster computation.",
    ],
    "ml_ai": [
        "Embeddings represent text as vectors so similar meanings are close.",
        "Overfitting happens when a model memorizes training data.",
        "Use cross-validation to estimate generalization performance.",
        "A confusion matrix summarizes classification errors.",
        "RAG combines retrieval with generation for grounded answers.",
        "Dimensionality reduction helps visualize high-dimensional data.",
    ],
    "weather": [
        "Monitor official advisories when a typhoon is nearby.",
        "Avoid driving through flooded roads during heavy rain.",
        "Storm surge can be more dangerous than wind in coastal zones.",
        "Prepare emergency supplies like water, food, and batteries.",
        "Secure loose objects before severe weather arrives.",
        "A tropical storm can intensify quickly over warm waters.",
    ],
    "business": [
        "A loyalty program can increase repeat purchases.",
        "Reduce churn by identifying early warning signals.",
        "Customer retention improves when support resolves issues quickly.",
        "Segment users to tailor messaging to their needs.",
        "Measure campaign impact with controlled experiments.",
        "Forecast demand to prevent stockouts during peak seasons.",
    ],
    "health": [
        "Walking daily can improve cardiovascular health.",
        "Breathing exercises can reduce stress in the short term.",
        "Prioritize sleep to support focus and memory.",
        "Balance meals with protein, fiber, and healthy fats.",
        "Avoid sugary drinks to reduce empty calories.",
        "Strength training supports bone density over time.",
    ],
}

# Expand by repeating + light perturbation (simple + stable for demos)
def expand_dataset(topics, repeats=12):
    rows = []
    for topic, base in topics.items():
        for _ in range(repeats):
            for s in base:
                rows.append((topic, s))
    df = pd.DataFrame(rows, columns=["topic", "text"]).drop_duplicates().reset_index(drop=True)
    df.insert(0, "doc_id", [f"D{i:04d}" for i in range(len(df))])
    return df

docs_df = expand_dataset(topics, repeats=10)
docs_df.shape, docs_df.head()

((36, 3),
   doc_id    topic  \
 0  D0000  finance   
 1  D0001  finance   
 2  D0002  finance   
 3  D0003  finance   
 4  D0004  finance   
 
                                                               text  
 0  Reduce spending by reviewing subscriptions and recurring bills.  
 1  Build an emergency fund by setting aside a small amount weekly.  
 2  Pay down high-interest debt first to save on interest payments.  
 3                 Banks assess credit risk before approving loans.  
 4                  Diversify investments to manage portfolio risk.  )

In [4]:
# 20 evaluation queries (covering multiple intents)
eval_queries = [
    ("Q01", "How can I cut back on expenses without feeling deprived?"),
    ("Q02", "What should I do first to pay down my debt?"),
    ("Q03", "How do banks decide whether to approve loans?"),
    ("Q04", "How can I reduce risk in my investment portfolio?"),
    
    ("Q05", "My program is slow. How do I speed it up?"),
    ("Q06", "How do I find bottlenecks in my code?"),
    ("Q07", "How can I make data processing faster in Python?"),
    ("Q08", "How do I make a system more reliable when network calls fail?"),
    
    ("Q09", "What is an embedding in NLP?"),
    ("Q10", "Why does my model perform well on train but poorly on test?"),
    ("Q11", "How do I evaluate a classifier's errors?"),
    ("Q12", "What is RAG and why would I use it?"),
    
    ("Q13", "A typhoon is coming. What should I prepare?"),
    ("Q14", "Is storm surge dangerous, and why?"),
    ("Q15", "Should I drive during floods?"),
    ("Q16", "How do storms get stronger over warm water?"),
    
    ("Q17", "How do I keep customers from leaving?"),
    ("Q18", "Does a loyalty program help retention?"),
    ("Q19", "How can I forecast demand to avoid stockouts?"),
    ("Q20", "What are quick ways to reduce stress?"),
]

queries_df = pd.DataFrame(eval_queries, columns=["query_id", "query"])
queries_df

,query_id,query
0,Q01,How can I cut back on expenses without feeling deprived?
1,Q02,What should I do first to pay down my debt?
2,Q03,How do banks decide whether to approve loans?
3,Q04,How can I reduce risk in my investment portfolio?
4,Q05,My program is slow. How do I speed it up?
5,Q06,How do I find bottlenecks in my code?
6,Q07,How can I make data processing faster in Python?
7,Q08,How do I make a system more reliable when network calls fail?
8,Q09,What is an embedding in NLP?
9,Q10,Why does my model perform well on train but poorly on test?


## 3) Define retrieval systems to compare

We’ll build two retrieval systems:
- **System A**: `all-MiniLM-L6-v2` (fast baseline)
- **System B**: `all-mpnet-base-v2` (often stronger, slower)

Both will use:
- normalized embeddings
- FAISS IndexFlatIP (inner product) → cosine-like ranking

You can later add more systems (e.g., different similarity metric, HNSW, etc.).


In [5]:
SYSTEMS = {
    "A_MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "B_MPNET": "sentence-transformers/all-mpnet-base-v2",
}

In [6]:
def build_system(model_name: str, texts: list[str]):
    model = SentenceTransformer(model_name)
    emb = model.encode(texts, normalize_embeddings=True).astype("float32")
    dim = emb.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(emb)
    return model, index, emb

texts = docs_df["text"].tolist()

systems = {}
for sys_name, model_name in SYSTEMS.items():
    model, index, emb = build_system(model_name, texts)
    systems[sys_name] = {"model_name": model_name, "model": model, "index": index}
    print(sys_name, "->", model_name, "| index size:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


A_MiniLM -> sentence-transformers/all-MiniLM-L6-v2 | index size: 36


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


B_MPNET -> sentence-transformers/all-mpnet-base-v2 | index size: 36


## 4) Retrieve top-10 for each query (for labeling)

We’ll store retrieval results in a table:

Columns:
- system
- query_id, query
- rank (1..10)
- doc_id, topic, text
- score

Then you manually label relevance:
- `label = 1` relevant, `0` not relevant
Optionally:
- `label = 2` highly relevant (if you want graded relevance later)

**For teaching:** have students label 3–5 queries each, then combine.


In [7]:
def retrieve(system_key: str, query: str, k=10):
    sys = systems[system_key]
    model = sys["model"]
    index = sys["index"]
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    
    t0 = time.time()
    scores, idx = index.search(q, k)
    latency_ms = (time.time() - t0) * 1000.0
    
    rows = []
    for rank, (i, score) in enumerate(zip(idx[0], scores[0]), start=1):
        rows.append({
            "system": system_key,
            "query": query,
            "rank": rank,
            "doc_id": docs_df.loc[i, "doc_id"],
            "topic": docs_df.loc[i, "topic"],
            "score": float(score),
            "text": docs_df.loc[i, "text"],
            "latency_ms": latency_ms,
        })
    return rows

retrieval_rows = []
for qid, q in eval_queries:
    for sys_key in systems.keys():
        rows = retrieve(sys_key, q, k=10)
        for r in rows:
            r["query_id"] = qid
        retrieval_rows.extend(rows)

retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_df.head(10)

,system,query,rank,doc_id,topic,score,text,latency_ms,query_id
0,A_MiniLM,How can I cut back on expenses without feeling deprived?,1,D0000,finance,0.477811,Reduce spending by reviewing subscriptions and recurring bills.,0.053167,Q01
1,A_MiniLM,How can I cut back on expenses without feeling deprived?,2,D0001,finance,0.453580,Build an emergency fund by setting aside a small amount weekly.,0.053167,Q01
2,A_MiniLM,How can I cut back on expenses without feeling deprived?,3,D0034,health,0.356995,Avoid sugary drinks to reduce empty calories.,0.053167,Q01
3,A_MiniLM,How can I cut back on expenses without feeling deprived?,4,D0002,finance,0.324181,Pay down high-interest debt first to save on interest payments.,0.053167,Q01
4,A_MiniLM,How can I cut back on expenses without feeling deprived?,5,D0005,finance,0.273534,Track your cash flow to understand where your money goes.,0.053167,Q01
5,A_MiniLM,How can I cut back on expenses without feeling deprived?,6,D0029,business,0.230379,Forecast demand to prevent stockouts during peak seasons.,0.053167,Q01
6,A_MiniLM,How can I cut back on expenses without feeling deprived?,7,D0021,weather,0.224458,"Prepare emergency supplies like water, food, and batteries.",0.053167,Q01
7,A_MiniLM,How can I cut back on expenses without feeling deprived?,8,D0031,health,0.221543,Breathing exercises can reduce stress in the short term.,0.053167,Q01
8,A_MiniLM,How can I cut back on expenses without feeling deprived?,9,D0008,software,0.210603,Add caching to avoid recomputing expensive results.,0.053167,Q01
9,A_MiniLM,How can I cut back on expenses without feeling deprived?,10,D0019,weather,0.198999,Avoid driving through flooded roads during heavy rain.,0.053167,Q01


### Labeling template

We’ll create a labeling table where you fill `label`.

**How to label**
- `1` = relevant to the query intent (would help answer the query)
- `0` = not relevant

Optional:
- `2` = highly relevant (best possible match)

For the metrics in this notebook, we’ll treat `label >= 1` as relevant.


In [8]:
# Create a labeling table (start with NaNs for labels)
label_df = retrieval_df.copy()
label_df["label"] = np.nan

label_df = label_df[["system", "query_id", "query", "rank", "doc_id", "topic", "score", "text", "label", "latency_ms"]]
label_df.head(12)

,system,query_id,query,rank,doc_id,topic,score,text,label,latency_ms
0,A_MiniLM,Q01,How can I cut back on expenses without feeling deprived?,1,D0000,finance,0.477811,Reduce spending by reviewing subscriptions and recurring bills.,NaN,0.053167
1,A_MiniLM,Q01,How can I cut back on expenses without feeling deprived?,2,D0001,finance,0.453580,Build an emergency fund by setting aside a small amount weekly.,NaN,0.053167
2,A_MiniLM,Q01,How can I cut back on expenses without feeling deprived?,3,D0034,health,0.356995,Avoid sugary drinks to reduce empty calories.,NaN,0.053167
3,A_MiniLM,Q01,How can I cut back on expenses without feeling deprived?,4,D0002,finance,0.324181,Pay down high-interest debt first to save on interest payments.,NaN,0.053167
4,A_MiniLM,Q01,How can I cut back on expenses without feeling deprived?,5,D0005,finance,0.273534,Track your cash flow to understand where your money goes.,NaN,0.053167
5,A_MiniLM,Q01,How can I cut back on expenses without feeling deprived?,6,D0029,business,0.230379,Forecast demand to prevent stockouts during peak seasons.,NaN,0.053167
6,A_MiniLM,Q01,How can I cut back on expenses without feeling deprived?,7,D0021,weather,0.224458,"Prepare emergency supplies like water, food, and batteries.",NaN,0.053167
7,A_MiniLM,Q01,How can I cut back on expenses without feeling deprived?,8,D0031,health,0.221543,Breathing exercises can reduce stress in the short term.,NaN,0.053167
8,A_MiniLM,Q01,How can I cut back on expenses without feeling deprived?,9,D0008,software,0.210603,Add caching to avoid recomputing expensive results.,NaN,0.053167
9,A_MiniLM,Q01,How can I cut back on expenses without feeling deprived?,10,D0019,weather,0.198999,Avoid driving through flooded roads during heavy rain.,NaN,0.053167


#### Option A (recommended for class): export CSV, label externally, then re-import

If you want students to label in Google Sheets / Excel:

1) Export:
```python
label_df.to_csv("labels_to_fill.csv", index=False)
```

2) Students fill `label` column with 0/1 (or 0/1/2)

3) Re-import:
```python
labeled = pd.read_csv("labels_to_fill.csv")
```

#### Option B: label directly in the notebook
For a quick demo, we’ll label a few queries using helper functions below.


In [9]:
# Helper: show a single query's retrieved results for labeling
def show_query(system, query_id):
    view = label_df[(label_df["system"] == system) & (label_df["query_id"] == query_id)].sort_values("rank")
    return view[["rank", "doc_id", "topic", "score", "text", "label"]]

show_query("A_MiniLM", "Q03")

,rank,doc_id,topic,score,text,label
40,1,D0003,finance,0.675525,Banks assess credit risk before approving loans.,NaN
41,2,D0005,finance,0.322297,Track your cash flow to understand where your money goes.,NaN
42,3,D0002,finance,0.318749,Pay down high-interest debt first to save on interest payments.,NaN
43,4,D0024,business,0.168048,A loyalty program can increase repeat purchases.,NaN
44,5,D0026,business,0.161294,Customer retention improves when support resolves issues quickly.,NaN
45,6,D0000,finance,0.158646,Reduce spending by reviewing subscriptions and recurring bills.,NaN
46,7,D0025,business,0.140743,Reduce churn by identifying early warning signals.,NaN
47,8,D0018,weather,0.125416,Monitor official advisories when a typhoon is nearby.,NaN
48,9,D0014,ml_ai,0.120815,Use cross-validation to estimate generalization performance.,NaN
49,10,D0006,software,0.114137,Refactor code to reduce technical debt and improve maintainability.,NaN


### Quick demo labeling (optional)

Below is a tiny labeling helper. You can set labels by doc_id.

Example:
```python
set_labels("A_MiniLM", "Q03", relevant_doc_ids=[...])
```
Everything else in top-10 becomes 0 for that query/system.


In [10]:
def set_labels(system, query_id, relevant_doc_ids):
    mask = (label_df["system"] == system) & (label_df["query_id"] == query_id)
    # default all to 0
    label_df.loc[mask, "label"] = 0
    # set relevant to 1
    rel_mask = mask & label_df["doc_id"].isin(relevant_doc_ids)
    label_df.loc[rel_mask, "label"] = 1

# DEMO: label a few queries quickly (you can edit these)
# Finance loan query should match credit risk/loans snippets
set_labels("A_MiniLM", "Q03", relevant_doc_ids=["D0003"])  # Banks assess credit risk...
set_labels("B_MPNET", "Q03", relevant_doc_ids=["D0003"])

# Typhoon prep should match advisories, supplies, secure objects
set_labels("A_MiniLM", "Q13", relevant_doc_ids=["D0012", "D0015", "D0016"])  # advisories, supplies, secure objects (ids may differ—adjust)
set_labels("B_MPNET", "Q13", relevant_doc_ids=["D0012", "D0015", "D0016"])

# Stress should match breathing
set_labels("A_MiniLM", "Q20", relevant_doc_ids=["D0015"])  # breathing (ids may differ—adjust)
set_labels("B_MPNET", "Q20", relevant_doc_ids=["D0015"])

# View one labeled query
show_query("A_MiniLM", "Q03")

,rank,doc_id,topic,score,text,label
40,1,D0003,finance,0.675525,Banks assess credit risk before approving loans.,1.0
41,2,D0005,finance,0.322297,Track your cash flow to understand where your money goes.,0.0
42,3,D0002,finance,0.318749,Pay down high-interest debt first to save on interest payments.,0.0
43,4,D0024,business,0.168048,A loyalty program can increase repeat purchases.,0.0
44,5,D0026,business,0.161294,Customer retention improves when support resolves issues quickly.,0.0
45,6,D0000,finance,0.158646,Reduce spending by reviewing subscriptions and recurring bills.,0.0
46,7,D0025,business,0.140743,Reduce churn by identifying early warning signals.,0.0
47,8,D0018,weather,0.125416,Monitor official advisories when a typhoon is nearby.,0.0
48,9,D0014,ml_ai,0.120815,Use cross-validation to estimate generalization performance.,0.0
49,10,D0006,software,0.114137,Refactor code to reduce technical debt and improve maintainability.,0.0


⚠️ **Note:** The quick demo labels above assume specific doc_ids.
Because we expanded the dataset deterministically, they should be stable,
but if you modify the dataset, update the doc_ids accordingly.

For real evaluation, you should label **all 20 queries × top-10** for each system.


## 5) Metrics

We’ll compute (per system):

### Precision@k
Out of top-k retrieved results, how many are relevant?
\[ Precision@k = \frac{\# relevant in top k}{k} \]

### Recall@k (requires a ground-truth relevant set)
Out of all relevant results, how many did we retrieve in top-k?
\[ Recall@k = \frac{\# relevant retrieved in top k}{\# relevant total} \]

In our setup, “relevant total” is approximated by:
- all items labeled relevant within top-10 (a *limited pool*), or
- a separate ground-truth set (better)

### MRR (Mean Reciprocal Rank)
Looks only at the **rank of the first relevant item**:
- if first relevant is at rank 1 → RR = 1
- rank 2 → RR = 1/2
- rank 5 → RR = 1/5
MRR is the average RR over queries.

MRR is easy to understand and very practical.


In [11]:
def precision_at_k(labels, k):
    labels_k = labels[:k]
    return np.mean(labels_k)

def reciprocal_rank(labels):
    # labels is a list/array of 0/1 relevance per rank (rank-ordered)
    for i, rel in enumerate(labels, start=1):
        if rel >= 1:
            return 1.0 / i
    return 0.0

def compute_metrics(labeled_df, k=5, recall_mode="pool"):
    """Compute metrics per system.
    
    recall_mode:
      - 'pool': treat #relevant in top-10 as the "total relevant" pool (limited)
      - 'none': skip recall
    """
    out_rows = []
    for system in labeled_df["system"].unique():
        sys_df = labeled_df[labeled_df["system"] == system]
        per_query = []
        recalls = []
        precs = []
        rrs = []
        latencies = []
        
        for qid in sys_df["query_id"].unique():
            qdf = sys_df[sys_df["query_id"] == qid].sort_values("rank")
            if qdf["label"].isna().any():
                # Skip unlabeled queries
                continue
            labels = qdf["label"].astype(int).to_numpy()  # top-10 labels
            latencies.append(float(qdf["latency_ms"].iloc[0]))
            
            precs.append(precision_at_k(labels, k))
            rrs.append(reciprocal_rank(labels))
            
            if recall_mode == "pool":
                total_rel = labels.sum()
                rel_at_k = labels[:k].sum()
                recalls.append(rel_at_k / total_rel if total_rel > 0 else 0.0)
        
        if len(precs) == 0:
            continue
        
        out_rows.append({
            "system": system,
            "Precision@{}".format(k): float(np.mean(precs)),
            "MRR": float(np.mean(rrs)),
            "AvgLatency(ms)": float(np.mean(latencies)),
            "Recall@{}".format(k): float(np.mean(recalls)) if recall_mode == "pool" else np.nan,
            "labeled_queries": len(precs),
        })
    return pd.DataFrame(out_rows)

metrics_k5 = compute_metrics(label_df, k=5, recall_mode="pool")
metrics_k10 = compute_metrics(label_df, k=10, recall_mode="pool")

metrics_k5, metrics_k10

(     system  Precision@5       MRR  AvgLatency(ms)  Recall@5  labeled_queries
 0  A_MiniLM     0.066667  0.333333        0.033617  0.333333                3
 1   B_MPNET     0.066667  0.333333        0.040213  0.333333                3,
      system  Precision@10       MRR  AvgLatency(ms)  Recall@10  \
 0  A_MiniLM      0.033333  0.333333        0.033617   0.333333   
 1   B_MPNET      0.033333  0.333333        0.040213   0.333333   
 
    labeled_queries  
 0                3  
 1                3  )

## 6) Before vs After comparison table

We’ll treat:
- System A as “before”
- System B as “after”

You can flip this depending on your narrative (speed vs quality).


In [12]:
def before_after(metrics_df, before="A_MiniLM", after="B_MPNET"):
    m = metrics_df.set_index("system")
    if before not in m.index or after not in m.index:
        return pd.DataFrame()
    cols = [c for c in metrics_df.columns if c not in ["system"]]
    rows = []
    for c in cols:
        b = m.loc[before, c]
        a = m.loc[after, c]
        if isinstance(b, (int, float, np.floating)) and isinstance(a, (int, float, np.floating)):
            delta = a - b
        else:
            delta = None
        rows.append({"metric": c, "before": b, "after": a, "delta(after-before)": delta})
    return pd.DataFrame(rows)

before_after_k5 = before_after(metrics_k5)
before_after_k10 = before_after(metrics_k10)

before_after_k5, before_after_k10

(            metric    before     after  delta(after-before)
 0      Precision@5  0.066667  0.066667             0.000000
 1              MRR  0.333333  0.333333             0.000000
 2   AvgLatency(ms)  0.033617  0.040213             0.006596
 3         Recall@5  0.333333  0.333333             0.000000
 4  labeled_queries  3.000000  3.000000                  NaN,
             metric    before     after  delta(after-before)
 0     Precision@10  0.033333  0.033333             0.000000
 1              MRR  0.333333  0.333333             0.000000
 2   AvgLatency(ms)  0.033617  0.040213             0.006596
 3        Recall@10  0.333333  0.333333             0.000000
 4  labeled_queries  3.000000  3.000000                  NaN)

## 7) Interpretation prompts

Use these prompts to make students think like engineers:

1. **If Precision@5 improves but latency worsens**, is that acceptable?
2. **If MRR improves**, what does that mean for user experience?
3. **If Recall@10 drops**, what kind of failures might users feel?
4. Which queries did System B improve? Which did it hurt?
5. Are you evaluating the right thing for your product (semantic relevance vs task usefulness)?

> **Important:** Metrics are only meaningful with good labels. Garbage labels → garbage metrics.


## 8) Next steps (what to do about evaluation in real systems)

- Create a **query log** from real users
- Sample queries weekly and label top-k
- Track metrics over time (regression detection)
- Add slice analysis:
  - short vs long queries
  - slang vs formal
  - domain A vs domain B
- Combine automatic metrics with human feedback:
  - thumbs up/down
  - click-through
  - “did you find what you wanted?”

This is how retrieval systems get better: **measurement + iteration**.


## Outputs checklist

- ✅ small table of metrics (Precision@k, Recall@k, MRR, latency)
- ✅ before vs after comparison (System A vs System B)
- ✅ labeling workflow for 20 queries × top-10 results
